# Delta Sharing - capa golden
Se comparten las tablas de KPIs de `golden` con un destinatario externo (open sharing).
Las tablas se agregan `WITH HISTORY` porque tienen deletion vectors activos (Databricks los habilita por defecto en las tablas nuevas) y Delta Sharing exige compartir el historial en ese caso.
El destinatario recibe un enlace de activacion para descargar sus credenciales y leer las tablas desde Power BI, pandas o Spark sin tener acceso al workspace.

In [ ]:
dbutils.widgets.text("catalogo", "catalog_prod")
dbutils.widgets.text("share", "share_instacart_golden")
dbutils.widgets.text("recipient", "recipient_area_comercial")

CATALOGO = dbutils.widgets.get("catalogo")
SHARE = dbutils.widgets.get("share")
RECIPIENT = dbutils.widgets.get("recipient")
TABLAS = ["kpi_departamentos", "top_productos", "actividad_dia_hora"]

## Share y tablas

In [ ]:
spark.sql(f"CREATE SHARE IF NOT EXISTS {SHARE} COMMENT 'KPIs de negocio Instacart (capa golden)'")

compartidas = {r.shared_object for r in spark.sql(f"SHOW ALL IN SHARE {SHARE}").collect()}
for tabla in TABLAS:
    nombre = f"{CATALOGO}.golden.{tabla}"
    if nombre in compartidas:
        print(f"Ya estaba en el share: {nombre}")
        continue
    spark.sql(f"ALTER SHARE {SHARE} ADD TABLE {nombre} WITH HISTORY")
    print(f"Agregada al share: {nombre}")

## Destinatario y permiso

In [ ]:
spark.sql(f"CREATE RECIPIENT IF NOT EXISTS {RECIPIENT} COMMENT 'Area comercial - consumo externo de KPIs'")
spark.sql(f"GRANT SELECT ON SHARE {SHARE} TO RECIPIENT {RECIPIENT}")
print(f"{RECIPIENT} tiene SELECT sobre {SHARE}")

## Validacion
En `DESCRIBE RECIPIENT` aparece el `activation_link` que se envia al destinatario.

In [ ]:
display(spark.sql(f"SHOW ALL IN SHARE {SHARE}"))

In [ ]:
display(spark.sql(f"DESCRIBE RECIPIENT {RECIPIENT}"))